# T5.5 EDL Cola (*Queue*)

## 1. Concepto

Una **cola** (*queue*) es una secuencia de elementos que se gestiona siguiendo un criterio **FIFO** (*First In, First Out*): el primer elemento en entrar es el primero en salir.

- Los elementos se **insertan** (*encolar* / *add*) siempre al **final** de la cola.
- Solo se pueden **extraer** (*desencolar* / *extract*) del **principio** de la cola.

<div align="center">
    <img src="fig/queue_concept.png" alt="Concepto de Cola" style="max-width:550px"/>
</div>

## 2. Interfaz

Una EDL tipo cola ofrece la siguiente interfaz de métodos:

| Método | Descripción |
|--------|-------------|
| `Queue()` | Crea una cola vacía |
| `add(e)` | Inserta el elemento `e` al final de la cola |
| `extract()` | Extrae y devuelve el primer elemento. Lanza excepción si vacía |
| `first()` | Devuelve el primer elemento sin extraerlo. Lanza excepción si vacía |
| `last()` | Devuelve el último elemento sin extraerlo. Lanza excepción si vacía |
| `empty()` | Indica si la cola está vacía |
| `size()` | Devuelve el número de elementos encolados |

Todas estas operaciones se pueden implementar con un coste temporal constante $\Theta(1)$.

## 3. Implementación con nodos enlazados: `QueueLinked`

La cola se implementa con una secuencia enlazada ordenada del primero al último. Se mantienen **dos referencias**: `__first` (para extraer) y `__last` (para insertar).

Mantendremos un atributo `__size` con el tamaño de la cola para poder implementar la operación `size()` en tiempo constante (sin necesidad de recorrer toda la secuencia).

Remarcar (como en el caso de `StackLinked`) que la clase `Node` será de uso interno.


<div align="center">
    <img src="fig/queue_linked.png" alt="Cola con nodos enlazados" style="max-width:650px"/>
</div>

In [7]:
class Node:
    """Nodo de una secuencia enlazada."""
    def __init__(self, data, next=None):
        self.data = data
        self.next = next


class QueueLinked:
    """Cola implementada con nodos enlazados.
    
    Atributos:
        __first (Node): referencia al primer nodo.
        __last (Node): referencia al último nodo.
        __size (int): número de elementos encolados.
    """
    
    def __init__(self):
        """Crea una cola vacía."""
        self.__first = None
        self.__last = None
        self.__size = 0
    
    def add(self, x):
        """Inserta el elemento x al final de la cola."""
        node = Node(x)
        if self.__last is not None:
            self.__last.next = node
        else:
            self.__first = node   # la cola estaba vacía
        self.__last = node
        self.__size += 1
    
    def extract(self):
        """Extrae y devuelve el primer elemento.
        Lanza IndexError si la cola está vacía."""
        if self.__size == 0:
            raise IndexError("Cola vacía")
        x = self.__first.data
        self.__first = self.__first.next
        if self.__first is None:
            self.__last = None    # la cola ahora está vacía
        self.__size -= 1
        return x
    
    def first(self):
        """Devuelve el primer elemento sin extraerlo."""
        if self.__size == 0:
            raise IndexError("Cola vacía")
        return self.__first.data
    
    def last(self):
        """Devuelve el último elemento sin extraerlo."""
        if self.__size == 0:
            raise IndexError("Cola vacía")
        return self.__last.data
    
    def empty(self):
        """Indica si la cola está vacía."""
        return self.__first is None
    
    def size(self):
        """Devuelve el número de elementos encolados."""
        return self.__size
    
    def __str__(self):
        result = "(QueueLinked) first → "
        aux = self.__first
        while aux is not None:
            result += f"[{aux.data}] → "
            aux = aux.next
        result += "None"
        return result

## 4. Implementación con array circular: `QueueArray`

Implementamos la cola mediante una lista Python de tamaño fijo (denotado por el atributo `__MAX`) referenciada por el atributo `__arr`. Dicha lista se usará como soporte del concepto de **array circular contínuo**:
- La posición de la cabeza de la cola (primer elemento) es marcada por el índice `__first`.
- La posición del final de la cola (último elemento) es marcada por el índice `__last`. 
- Extraer un elemento implica incrementar el índice `__first` en una unidad.
- La posición del array que sigue a `__MAX-1` es la `0`. 
- Eventualmente, `__first` puede ser mayor que `__last`.
- Necesitamos un atributo `__size` para conocer el número de elementos encolados y para distinguir entre una cola **llena** y una **vacía** cuando `__first > __last` en el array circular.


<div align="center">
    <img src="fig/queue_circular_array.png" alt="Cola con array circular" style="max-width:550px"/>
</div>

<div align="center">
    <img src="fig/queue_circular_array_2.png" alt="Cola con array circular" style="max-width:550px"/>
</div>

> Nota: el atributo `__MAX` no es estrictamente necesario (se puede calcular con `len(__arr)`), pero es una buena práctica considerarlo, ya que estamos imitando un contexto de uso de arrays puros, como sucedería con C/C++.

In [8]:
class QueueArray:
    """Cola implementada con un array circular.
    
    Atributos:
        __arr (list): array circular.
        __first (int): índice del primer elemento.
        __last (int): índice del último elemento.
        __size (int): número de elementos encolados.
        __MAX (int): capacidad máxima.
    """
    
    def __init__(self, max_size=100):
        """Crea una cola vacía con capacidad máxima max_size."""
        self.__MAX = max_size
        self.__arr = [None] * self.__MAX
        self.__first = 0
        self.__last = -1
        self.__size = 0
    
    def __increment(self, i):
        """Devuelve la siguiente posición en el array circular."""
        return (i + 1) % self.__MAX
    
    def add(self, x):
        """Inserta x al final. Lanza IndexError si la cola está llena."""
        if self.__size == self.__MAX:
            raise IndexError("Cola llena")
        self.__last = self.__increment(self.__last)
        self.__arr[self.__last] = x
        self.__size += 1
    
    def extract(self):
        """Extrae y devuelve el primero. Lanza IndexError si vacía."""
        if self.__size == 0:
            raise IndexError("Cola vacía")
        x = self.__arr[self.__first]
        self.__first = self.__increment(self.__first)
        self.__size -= 1
        return x
    
    def first(self):
        """Devuelve el primer elemento sin extraerlo."""
        if self.__size == 0:
            raise IndexError("Cola vacía")
        return self.__arr[self.__first]
    
    def last(self):
        """Devuelve el último elemento sin extraerlo."""
        if self.__size == 0:
            raise IndexError("Cola vacía")
        return self.__arr[self.__last]
    
    def empty(self):
        return self.__size == 0
    
    def size(self):
        return self.__size
    
    def __str__(self):
        result = "(QueueArray) first → "
        if self.__size > 0:
            i = self.__first
            for _ in range(self.__size):
                result += f"[{self.__arr[i]}] → "
                i = self.__increment(i)
        result += "None"
        return result

## 5. Uso y demostración

Ambas implementaciones ofrecen la **misma interfaz**, por lo que el código de uso es idéntico. Alternar entre una y otra resulta trivial:

In [9]:
# Podemos usar cualquiera de las dos implementaciones:

cola = QueueLinked() # Comentar/Descomentar
#cola = QueueArray() # Comentar/Descomentar

cola.add(14)
cola.add(33)

print(f"first(): {cola.first()}")   # 14
print(f"last():  {cola.last()}")    # 33
print(f"size():  {cola.size()}")    # 2

print("\nExtrayendo todos los elementos:")
while not cola.empty():
    print(f"  extract(): {cola.extract()}")  # 14, luego 33

print(f"empty(): {cola.empty()}")  # True

# Intentar extraer de una cola genera una excepción
try:
    cola.extract()
except IndexError as e:
    print(f"Error: {e}")

print("\nNueva cola desde una lista:")
cola = QueueLinked()
for v in [10, 13, 22, 8]:
    cola.add(v)
print(cola)

print(f"\nextract(): {cola.extract()}")
print(cola)

first(): 14
last():  33
size():  2

Extrayendo todos los elementos:
  extract(): 14
  extract(): 33
empty(): True
Error: Cola vacía

Nueva cola desde una lista:
(QueueLinked) first → [10] → [13] → [22] → [8] → None

extract(): 10
(QueueLinked) first → [13] → [22] → [8] → None


Vamos a comprobar la circularidad del array subyacente:

In [10]:
cola = QueueArray(max_size=5)

# Llenamos la cola
for v in [10, 20, 30, 40, 50]:
    cola.add(v)
print("Cola llena:", cola)
print(f"  _first={cola._QueueArray__first}, _last={cola._QueueArray__last}, _size={cola._QueueArray__size}")

# Extraemos 3 elementos
for _ in range(3):
    cola.extract()
print("\nTras 3 extracciones:", cola)
print(f"  _first={cola._QueueArray__first}, _last={cola._QueueArray__last}, _size={cola._QueueArray__size}")

# Añadimos 2 más (reutiliza posiciones 0 y 1)
cola.add(60)
cola.add(70)
print("\nTras añadir 60 y 70:", cola)
print(f"  _first={cola._QueueArray__first}, _last={cola._QueueArray__last}, _size={cola._QueueArray__size}")
print(f"  Array interno: {cola._QueueArray__arr}")

Cola llena: (QueueArray) first → [10] → [20] → [30] → [40] → [50] → None
  _first=0, _last=4, _size=5

Tras 3 extracciones: (QueueArray) first → [40] → [50] → None
  _first=3, _last=4, _size=2

Tras añadir 60 y 70: (QueueArray) first → [40] → [50] → [60] → [70] → None
  _first=3, _last=1, _size=4
  Array interno: [60, 70, 30, 40, 50]


## 6. Comparativa de implementaciones

| Aspecto | Nodos enlazados (`QueueLinked`) | Array circular (`QueueArray`) |
|---------|-------------------------------|-------------------------------|
| **Complejidad temporal** | Todas: $\Theta(1)$ | Todas: $\Theta(1)$ |
| **Memoria** | Variable (crece/decrece dinámicamente) + sobrecoste por referencias (despreciable) | Fija (reserva `MAX` posiciones) |
| **Talla máxima** | Sin límite | Limitada a `MAX` |

Al igual como con `StackArray`, la implementación con arrays presenta el inconveniente de la estimación del tamaño máximo. 

Se podría implementar una variante de `add()` que permita redimensionar automáticamente el array, como vimos en la [Sección 4.1 del T5.3 sobre gestión de secuencias con memoria contigua](./T5.3%20Gestión%20de%20secuencias%20con%20memoria%20contigua.ipynb), pero entonces, el coste de esa operación pasaria a ser $\Omega(1) \cap O(n)$.